# 🚀 PlasticSense AI - YOLOv11 Model Training

### 🌟 Overview
This notebook represents the core of the PlasticSense AI project: training a state-of-the-art YOLOv11 object detection model. We will leverage the highly curated, augmented dataset we built over the last 5 notebooks to train a model capable of detecting our 8 target plastic categories in real-world environments.

### 🧠 Transfer Learning
To drastically reduce training time and improve accuracy, we will not train from scratch. We will initialize our model using the official COCO-pretrained weights (`yolo11s.pt`) and fine-tune it for our specific domain.

### 📈 Advanced Features
This pipeline implements Mixed Precision Training, Early Stopping, Checkpoint Resumption (in case Colab disconnects), and automated export of metrics.

## 1. Environment Setup & Library Imports
Ensure the GPU is active and all required libraries (especially Ultralytics) are installed.

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib pillow tensorboard

In [ ]:
import os
import shutil
import json
import yaml
import time
import torch
import logging
from pathlib import Path
from typing import Dict, Any

import pandas as pd
import matplotlib.pyplot as plt
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

console = Console()

## 2. Hardware Verification
YOLOv11 requires substantial compute. We verify the presence of a CUDA-capable GPU, warning the user if fallback to CPU is imminent.

In [ ]:
def check_hardware():
    table = Table(title="Hardware & Environment Status", show_header=True)
    table.add_column("Component", style="cyan")
    table.add_column("Status/Version", justify="right")
    
    # PyTorch
    table.add_row("PyTorch Version", torch.__version__)
    
    # GPU
    has_gpu = torch.cuda.is_available()
    if has_gpu:
        device_name = torch.cuda.get_device_name(0)
        cuda_ver = torch.version.cuda
        mem_allocated = torch.cuda.memory_allocated(0) / 1024**3
        mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        table.add_row("GPU Accelerator", f"[green]✔ {device_name}[/green]")
        table.add_row("CUDA Version", str(cuda_ver))
        table.add_row("GPU Memory", f"{mem_allocated:.2f} GB / {mem_total:.2f} GB")
    else:
        table.add_row("GPU Accelerator", "[bold red]✖ CPU ONLY (WARNING)[/bold red]")
        console.print("[bold red]WARNING: Training on CPU will take days. Please enable GPU in Colab (Runtime -> Change runtime type).[/bold red]")
        
    import ultralytics
    table.add_row("Ultralytics Version", ultralytics.__version__)
    
    console.print(table)
    return "cuda:0" if has_gpu else "cpu"

DEVICE = check_hardware()

## 3. Verify Augmented Dataset
Confirm that the `augmented_yolo` dataset and its YAML configuration exist.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
DATASET_DIR = PROJECT_ROOT / "datasets/augmented_yolo"
DATASET_YAML = DATASET_DIR / "dataset.yaml"

MODELS_DIR = PROJECT_ROOT / "models"
TRAIN_LOGS_DIR = MODELS_DIR / "train_logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOGS_DIR.mkdir(parents=True, exist_ok=True)

def verify_dataset():
    if not DATASET_YAML.exists():
        raise FileNotFoundError(f"Dataset YAML not found at {DATASET_YAML}. Run Notebook 05 first.")
    
    with open(DATASET_YAML, 'r') as f:
        config = yaml.safe_load(f)
        
    num_classes = len(config.get('names', {}))
    console.print(f"[green]✔ Dataset YAML verified. Found {num_classes} classes.[/green]")
    return config

dataset_config = verify_dataset()

## 4. Hyperparameter & Training Configuration
These variables control the learning dynamics of YOLOv11. Modify these based on the capabilities of your current GPU.

In [ ]:
# --- MODEL CONFIGURATION ---
MODEL_VARIANT = "yolo11s.pt" # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt

# --- TRAINING HYPERPARAMETERS ---
EPOCHS = 100
BATCH_SIZE = 16
IMAGE_SIZE = 640
PATIENCE = 20
OPTIMIZER = "AdamW"
SEED = 42

# --- ADVANCED FEATURES ---
MIXED_PRECISION = True  # Uses AMP for faster GPU training
CACHE_IMAGES = False    # Set to True if RAM allows (Speeds up epoch time)
WORKERS = 8             # CPU workers for dataloading

console.print(Panel.fit(
    f"[bold cyan]Training Configuration:[/bold cyan]\n"
    f"Model: {MODEL_VARIANT}\n"
    f"Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | Image Size: {IMAGE_SIZE}\n"
    f"Optimizer: {OPTIMIZER} | Patience: {PATIENCE}\n"
    f"Device: {DEVICE} | Mixed Precision: {MIXED_PRECISION}"
))

## 5. Model Initialization & Checkpoint Resumption
If a previous training run was disconnected halfway, the code automatically locates `last.pt` and resumes seamlessly.

In [ ]:
def initialize_model(model_variant: str, logs_dir: Path) -> YOLO:
    project_name = "PlasticSense_YOLOv11"
    project_dir = logs_dir / project_name
    
    # Check for existing checkpoint to resume
    last_checkpoint = project_dir / "train" / "weights" / "last.pt"
    
    if last_checkpoint.exists():
        console.print(f"[bold yellow]⚠ Found existing checkpoint at {last_checkpoint}. Initiating RESUME sequence...[/bold yellow]")
        model = YOLO(str(last_checkpoint))
        resume = True
    else:
        console.print(f"[bold green]✔ Initializing fresh transfer learning run using {model_variant}[/bold green]")
        model = YOLO(model_variant)
        resume = False
        
    # Print model parameters summary
    model.info()
    
    return model, resume, project_name

model, should_resume, PROJECT_NAME = initialize_model(MODEL_VARIANT, TRAIN_LOGS_DIR)

## 6. Execution: Train YOLOv11 Model
The training loop begins. Ultralytics automatically handles metric calculation, gradient scaling, learning rate scheduling, and early stopping.

In [ ]:
import datetime

start_time = time.time()
start_time_fmt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
console.print(f"\n[bold green]--- Training Initiated at {start_time_fmt} ---[/bold green]\n")

try:
    results = model.train(
        data=str(DATASET_YAML),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        patience=PATIENCE,
        optimizer=OPTIMIZER,
        seed=SEED,
        device=DEVICE,
        amp=MIXED_PRECISION,
        cache=CACHE_IMAGES,
        workers=WORKERS,
        project=str(TRAIN_LOGS_DIR),
        name=PROJECT_NAME,
        resume=should_resume,
        exist_ok=True,       # Allows appending to the same folder on resume
        plots=True,          # Automatically generate and save plots
        save=True,           # Save checkpoints
        val=True             # Validate during training
    )
except torch.cuda.OutOfMemoryError:
    console.print("[bold red]CRITICAL OOM ERROR: GPU ran out of memory![/bold red]")
    console.print("[yellow]Action: Please decrease BATCH_SIZE (e.g. to 8 or 4) or IMAGE_SIZE (e.g. to 416) and rerun.[/yellow]")
    raise
except KeyboardInterrupt:
    console.print("[bold yellow]Training Interrupted by User. Checkpoints are safely stored.[/bold yellow]")

end_time = time.time()
training_duration = end_time - start_time
hours, rem = divmod(training_duration, 3600)
minutes, seconds = divmod(rem, 60)
console.print(f"\n[bold green]--- Training Completed in {int(hours)}h {int(minutes)}m {int(seconds)}s ---[/bold green]\n")

## 7. Model Export & Architecture Export
We systematically extract `best.pt`, `results.csv`, and training parameters out of the nested log folders into the root `models/` directory for easy access.

In [ ]:
FINAL_MODEL_DIR = TRAIN_LOGS_DIR / PROJECT_NAME
BEST_PT_PATH = FINAL_MODEL_DIR / "weights" / "best.pt"
RESULTS_CSV = FINAL_MODEL_DIR / "results.csv"

if BEST_PT_PATH.exists():
    # Copy best weights to root models/
    shutil.copy2(BEST_PT_PATH, MODELS_DIR / "best.pt")
    console.print("[green]✔ best.pt saved to {MODELS_DIR / 'best.pt'}[/green]")
    
    # Export Hyperparameters and Config
    export_config = {
        "Model Variant": MODEL_VARIANT,
        "Epochs Trained": EPOCHS,
        "Batch Size": BATCH_SIZE,
        "Image Size": IMAGE_SIZE,
        "Optimizer": OPTIMIZER,
        "Training Time (s)": round(training_duration, 2),
        "Dataset Path": str(DATASET_DIR),
        "Device Used": DEVICE
    }
    
    with open(MODELS_DIR / "training_configuration.json", 'w') as f:
        json.dump(export_config, f, indent=4)
        
    pd.DataFrame([export_config]).to_csv(MODELS_DIR / "training_configuration.csv", index=False)
    console.print("[green]✔ Training configuration and hyperparameters saved.[/green]")
else:
    console.print("[bold red]✖ Training failed to produce best.pt[/bold red]")

## 8. Training Curves & Confusion Matrix Visualization
Displaying publication-quality graphs generated automatically by Ultralytics during the training process.

In [ ]:
def display_training_metrics(metrics_dir: Path):
    plots = {
        "Training Curves": metrics_dir / "results.png",
        "Confusion Matrix": metrics_dir / "confusion_matrix.png",
        "F1 Curve": metrics_dir / "F1_curve.png",
        "Precision-Recall Curve": metrics_dir / "PR_curve.png"
    }
    
    for title, plot_path in plots.items():
        if plot_path.exists():
            try:
                img = cv2.imread(str(plot_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.figure(figsize=(15, 10))
                plt.imshow(img)
                plt.title(title, fontweight='bold', fontsize=16)
                plt.axis('off')
                plt.tight_layout()
                plt.show()
            except Exception as e:
                console.print(f"[yellow]Could not render {title}: {e}[/yellow]")
        else:
            console.print(f"[yellow]Plot not found: {title}[/yellow]")

display_training_metrics(FINAL_MODEL_DIR)

## 9. Performance Reports Generation
We parse the `results.csv` generated by YOLO to extract the highest achieved validation metrics and save them to explicit JSON/CSV files.

In [ ]:
if RESULTS_CSV.exists():
    df_results = pd.read_csv(RESULTS_CSV)
    df_results.columns = df_results.columns.str.strip() # Remove trailing spaces from column names
    
    try:
        best_epoch = df_results['metrics/mAP50-95(B)'].idxmax()
        best_metrics = df_results.iloc[best_epoch]
        
        model_report = {
            "Total Epochs Completed": len(df_results),
            "Best Epoch": int(best_metrics['epoch']),
            "Best Precision": round(best_metrics.get('metrics/precision(B)', 0.0), 4),
            "Best Recall": round(best_metrics.get('metrics/recall(B)', 0.0), 4),
            "Best mAP@50": round(best_metrics.get('metrics/mAP50(B)', 0.0), 4),
            "Best mAP@50-95": round(best_metrics.get('metrics/mAP50-95(B)', 0.0), 4)
        }
        
        with open(MODELS_DIR / "training_report.json", 'w') as f:
            json.dump(model_report, f, indent=4)
            
        pd.DataFrame([model_report]).to_csv(MODELS_DIR / "training_report.csv", index=False)
        console.print("[green]✔ Final metrics calculated and reports generated.[/green]")
    except KeyError as e:
        console.print(f"[red]Error parsing results.csv columns: {e}[/red]")
        model_report = {}
else:
    console.print("[red]✖ results.csv not found, cannot generate final metrics.[/red]")
    model_report = {}

## 10. Final Model Summary
Presenting the training culmination and moving to the next pipeline stage.

In [ ]:
console.print(Panel.fit(
    f"[bold green]Model Successfully Trained & Checkpointed[/bold green]\n\n"
    f"[bold cyan]✔ Total Epochs:[/bold cyan] {model_report.get('Total Epochs Completed', 'N/A')}\n"
    f"[bold cyan]✔ Best Epoch:[/bold cyan] {model_report.get('Best Epoch', 'N/A')}\n"
    f"[bold magenta]✔ Best Precision:[/bold magenta] {model_report.get('Best Precision', 'N/A')}\n"
    f"[bold magenta]✔ Best Recall:[/bold magenta] {model_report.get('Best Recall', 'N/A')}\n"
    f"[bold yellow]✔ Best mAP@50:[/bold yellow] {model_report.get('Best mAP@50', 'N/A')}\n"
    f"[bold yellow]✔ Best mAP@50-95:[/bold yellow] {model_report.get('Best mAP@50-95', 'N/A')}\n\n"
    f"[bold]Training Time:[/bold] {int(hours)}h {int(minutes)}m {int(seconds)}s\n"
    f"[bold]Model Saved Location:[/bold] PlasticSense_AI/models/best.pt\n\n"
    f"[bold green]Ready for Evaluation![/bold green]\n"
    f"[bold red]Next Notebook:[/bold red] 07_Model_Evaluation.ipynb"
))